In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/superstore_Task24_Dataset.csv", encoding="latin1")
print(df.shape)
df.info()
df.head()

(10800, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10800 entries, 0 to 10799
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10800 non-null  object 
 1   Order ID       10800 non-null  object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9983 non-null   float64
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


In [3]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report = missing_report[missing_report.missing_count > 0].sort_values("missing_count", ascending=False)
missing_report

,missing_count,missing_pct
Postal Code,817,7.56
Ship Date,806,7.46
Order Date,806,7.46
Ship Mode,806,7.46
Customer ID,806,7.46
Segment,806,7.46
Customer Name,806,7.46
Country,806,7.46
City,806,7.46
State,806,7.46


In [4]:
full_dup = df.duplicated().sum()
key_dup = df.duplicated(subset=["Order ID", "Product ID"]).sum()
rowid_dup = df["Row ID"].duplicated().sum()
print("Full row duplicates:", full_dup)
print("Order ID + Product ID duplicates:", key_dup)
print("Row ID duplicates:", rowid_dup)

Full row duplicates: 504
Order ID + Product ID duplicates: 512
Row ID duplicates: 799


In [5]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

range_issues = {
    "Sales <= 0": (df["Sales"] <= 0).sum(),
    "Quantity <= 0": (df["Quantity"] <= 0).sum(),
    "Discount < 0 or > 1": ((df["Discount"] < 0) | (df["Discount"] > 1)).sum(),
    "Ship Date < Order Date": (df["Ship Date"] < df["Order Date"]).sum(),
    "Invalid Order Date": df["Order Date"].isna().sum(),
    "Invalid Ship Date": df["Ship Date"].isna().sum(),
}
pd.Series(range_issues)

,0
Sales <= 0,0
Quantity <= 0,0
Discount < 0 or > 1,0
Ship Date < Order Date,0
Invalid Order Date,806
Invalid Ship Date,806


In [6]:
# (Spelling/Case problems)
for col in ["Ship Mode", "Segment", "Region", "Category", "Sub-Category", "Country"]:
    print(col, df[col].value_counts(dropna=False).to_dict(), "\n")

# Extra spaces / case issues
for col in df.select_dtypes("object").columns:
    n = (df[col].dropna() != df[col].dropna().str.strip()).sum()
    if n: print(f"{col}: {n} values with extra spaces")

#  Customer ID  multiple names
multi = df.groupby("Customer ID")["Customer Name"].nunique()
print("Customer IDs with >1 name:", (multi > 1).sum())

# Sub-Category duplicates
sc = df.groupby("Sub-Category")["Category"].nunique()
print("Sub-Categories in multiple Categories:", (sc > 1).sum())

Ship Mode {'Standard Class': 5968, 'Second Class': 1945, 'First Class': 1538, nan: 806, 'Same Day': 543} 

Segment {'Consumer': 5191, 'Corporate': 3020, 'Home Office': 1783, nan: 806} 

Region {'West': 3203, 'East': 2848, 'Central': 2323, 'South': 1620, nan: 806} 

Category {'Office Supplies': 6026, 'Furniture': 2121, 'Technology': 1847, nan: 806} 

Sub-Category {'Binders': 1523, 'Paper': 1370, 'Furnishings': 957, 'Phones': 889, 'Storage': 846, nan: 806, 'Art': 796, 'Accessories': 775, 'Chairs': 617, 'Appliances': 466, 'Labels': 364, 'Tables': 319, 'Envelopes': 254, 'Bookcases': 228, 'Fasteners': 217, 'Supplies': 190, 'Machines': 115, 'Copiers': 68} 

Country {'United States': 9994, nan: 806} 

Customer IDs with >1 name: 0
Sub-Categories in multiple Categories: 0


In [7]:
def audit(df):
    d = df.copy()
    d["Order Date"] = pd.to_datetime(d["Order Date"], errors="coerce")
    d["Ship Date"] = pd.to_datetime(d["Ship Date"], errors="coerce")
    rules = {
        "Missing values (any column)": d.isna().any(axis=1),
        "Full row duplicate": d.duplicated(keep="first"),
        "Sales <= 0": d["Sales"] <= 0,
        "Quantity <= 0": d["Quantity"] <= 0,
        "Discount out of 0-1": (d["Discount"] < 0) | (d["Discount"] > 1),
        "Ship Date before Order Date": d["Ship Date"] < d["Order Date"],
    }
    summary = pd.DataFrame({
        "Issue": list(rules.keys()),
        "Count": [int(m.sum()) for m in rules.values()],
    })
    summary["Pct of rows"] = (summary["Count"] / len(d) * 100).round(2)
    return summary, rules

summary, rules = audit(df)
summary

,Issue,Count,Pct of rows
0,Missing values (any column),817,7.56
1,Full row duplicate,504,4.67
2,Sales <= 0,0,0.00
3,Quantity <= 0,0,0.00
4,Discount out of 0-1,0,0.00
5,Ship Date before Order Date,0,0.00


In [8]:
log_rows = []
for issue, mask in rules.items():
    idx = df.index[mask.fillna(False)]
    for i in idx:
        log_rows.append({"Row ID": df.loc[i, "Row ID"], "Issue": issue})

issue_log = pd.DataFrame(log_rows)
print(issue_log.shape)
issue_log.head()

(1321, 2)


,Row ID,Issue
0,2235,Missing values (any column)
1,5275,Missing values (any column)
2,8799,Missing values (any column)
3,9147,Missing values (any column)
4,9148,Missing values (any column)


In [15]:

id_cols = ["Row ID", "Order ID"]
data_cols = [c for c in df.columns if c not in id_cols]
blank_mask = df[data_cols].isna().all(axis=1)
print("Fully blank rows:", blank_mask.sum())


clean = df[~blank_mask].copy()
print("After removing blank rows:", clean.shape)

print(clean.isna().sum()[clean.isna().sum() > 0])

clean.loc[clean["Postal Code"].isna() & (clean["City"] == "Burlington"), "Postal Code"] = 5401
print("Postal Code missing now:", clean["Postal Code"].isna().sum())   # 0


print("Duplicates:", clean.duplicated().sum())
print("Duplicates (ignoring Row ID):", clean.drop(columns="Row ID").duplicated().sum())

Fully blank rows: 806
After removing blank rows: (9994, 21)
Postal Code    11
dtype: int64
Postal Code missing now: 0
Duplicates: 0
Duplicates (ignoring Row ID): 1


In [13]:
from google.colab import files

summary.to_csv("audit_summary.csv", index=False)
issue_log.to_csv("issue_log.csv", index=False)
clean.to_csv("cleaned_sample.csv", index=False)

files.download("audit_summary.csv")
files.download("issue_log.csv")
files.download("cleaned_sample.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
clean["Order Date"] = pd.to_datetime(clean["Order Date"], format="%m/%d/%Y", errors="coerce")
clean["Ship Date"]  = pd.to_datetime(clean["Ship Date"],  format="%m/%d/%Y", errors="coerce")

range_issues = {
    "Sales <= 0": (clean["Sales"] <= 0).sum(),
    "Quantity <= 0": (clean["Quantity"] <= 0).sum(),
    "Discount < 0 or > 1": ((clean["Discount"] < 0) | (clean["Discount"] > 1)).sum(),
    "Ship Date < Order Date": (clean["Ship Date"] < clean["Order Date"]).sum(),
    "Invalid Order Date": clean["Order Date"].isna().sum(),
    "Invalid Ship Date": clean["Ship Date"].isna().sum(),
    "Negative Profit (info only)": (clean["Profit"] < 0).sum(),
}
print(pd.Series(range_issues))

print(clean["Order Date"].min(), clean["Order Date"].max())
print(clean[["Sales", "Quantity", "Discount", "Profit"]].describe())

Sales <= 0                        0
Quantity <= 0                     0
Discount < 0 or > 1               0
Ship Date < Order Date            0
Invalid Order Date                0
Invalid Ship Date                 0
Negative Profit (info only)    1871
dtype: int64
2015-01-03 00:00:00 2018-12-30 00:00:00
              Sales     Quantity     Discount       Profit
count   9994.000000  9994.000000  9994.000000  9994.000000
mean     229.858001     3.789574     0.156203    28.656896
std      623.245101     2.225110     0.206452   234.260108
min        0.444000     1.000000     0.000000 -6599.978000
25%       17.280000     2.000000     0.000000     1.728750
50%       54.490000     3.000000     0.200000     8.666500
75%      209.940000     5.000000     0.200000    29.364000
max    22638.480000    14.000000     0.800000  8399.976000


In [17]:
from google.colab import files

# 1. Duplicate remove
cols = [c for c in clean.columns if c != "Row ID"]
dup_mask = clean.duplicated(subset=cols, keep="first")
dup_ids = clean.loc[dup_mask, "Row ID"].tolist()
clean = clean[~dup_mask].copy()
print("Final clean shape:", clean.shape)   # (9993, 21)

# 2. Issue log (row-wise)
log = []
for rid in df.loc[blank_mask, "Row ID"]:
    log.append({"Row ID": rid, "Column": "All (except IDs)", "Issue": "Fully blank row",
                "Severity": "High", "Action": "Removed"})

pc_ids = df.loc[df["Postal Code"].isna() & (df["City"] == "Burlington"), "Row ID"]
for rid in pc_ids:
    log.append({"Row ID": rid, "Column": "Postal Code", "Issue": "Missing value",
                "Severity": "Low", "Action": "Filled with 05401 (Burlington, VT)"})

for rid in dup_ids:
    log.append({"Row ID": rid, "Column": "All (except Row ID)", "Issue": "Duplicate record",
                "Severity": "Medium", "Action": "Removed (kept first occurrence)"})

for rid in clean.loc[clean["Profit"] < 0, "Row ID"]:
    log.append({"Row ID": rid, "Column": "Profit", "Issue": "Negative profit",
                "Severity": "Info", "Action": "Retained (valid loss-making order)"})

issue_log = pd.DataFrame(log)

# 3. Audit summary
total = len(df)
summary = pd.DataFrame([
    ["Fully blank rows", int(blank_mask.sum()), "High", "Removed"],
    ["Missing Postal Code", len(pc_ids), "Low", "Filled (05401)"],
    ["Duplicate records", len(dup_ids), "Medium", "Removed"],
    ["Range violations (Sales, Qty, Discount, Dates)", 0, "Passed", "None"],
    ["Consistency issues (spelling, ID mapping)", 0, "Passed", "None"],
    ["Negative profit orders", int((clean["Profit"] < 0).sum()), "Info", "Retained"],
], columns=["Issue", "Count", "Severity", "Action"])
summary["Pct of original rows"] = (summary["Count"] / total * 100).round(2)
print(summary)

# 4. Save + download
summary.to_csv("audit_summary.csv", index=False)
issue_log.to_csv("issue_log.csv", index=False)
clean.to_csv("cleaned_sample.csv", index=False)
for f in ["audit_summary.csv", "issue_log.csv", "cleaned_sample.csv"]:
    files.download(f)

Final clean shape: (9993, 21)
                                            Issue  Count Severity  \
0                                Fully blank rows    806     High   
1                             Missing Postal Code     11      Low   
2                               Duplicate records      1   Medium   
3  Range violations (Sales, Qty, Discount, Dates)      0   Passed   
4       Consistency issues (spelling, ID mapping)      0   Passed   
5                          Negative profit orders   1870     Info   

           Action  Pct of original rows  
0         Removed                  7.46  
1  Filled (05401)                  0.10  
2         Removed                  0.01  
3            None                  0.00  
4            None                  0.00  
5        Retained                 17.31  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>